# IB / ORB break-and-excursion analysis, conditioned on market regime

Combines the IB and ORB studies from the project and reports every statistic
**per regime bin** — where "regime" is the daily volatility state from the
Market Regime work (rolling `sd_L` vs a 10-year baseline `baseSD`).

Three conditioners, read straight off the y-axis of the regime chart:

| conditioner | bins |
|---|---|
| **ratio** (`sd_L / baseSD`) | `<0.5, 0.5-1, 1-1.5, 1.5-2, 2.0+`  (top pooled — ratio caps ~2 on a 10y NQ baseline) |
| **percentile** (`sd_L` vs its own 10y history) | deciles `0-10 … 90-100` (not equal-count by design) |
| **overall** (sign of the ratio's 5-day change) | COMPRESSING / STABLE / EXPANDING |

Every table shows one row per bin + an `ALL(regime)` row (regime-valid window)
and an `ALL(hist)` row (full event history), with `n` on every row. Bins under
`MIN_N` print `(below min sample)` instead of a number. Session day is `+6h`
(the corrected, non-forward-looking definition), shared by regime and events.

Edit `REGIME_LOOKBACK` to swap the vol window (10/20/50). Bin edges are config,
so the same notebook run on an ES file (where ratio reaches ~5) fills the tails.

In [ ]:
# ============================ CONFIG ============================
import numpy as np, pandas as pd
from IPython.display import display, Markdown

DATA_PATH = r"C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv"

BASELINE_YEARS      = 10.0         # regime baseline; trailLen = round(BASELINE_YEARS*252)
BARS_PER_YEAR       = 252
REGIME_LOOKBACK     = 20           # primary vol window (also computes 10 & 50)
TREND_WINDOW        = 5            # for the ratio's change -> COMPRESSING/EXPANDING
TREND_THRESHOLD     = 0.05
SESSION_SHIFT_HOURS = 6            # +6h session day

MIN_N               = 100          # gate for a bin to print stats

# bin edges (config so ES/NQ both work)
RATIO_EDGES  = [0, 0.5, 1.0, 1.5, 2.0, np.inf]
RATIO_LABELS = ['<0.5', '0.5-1', '1-1.5', '1.5-2', '2.0+']
PCT_EDGES    = list(range(0, 101, 10))          # deciles
OVERALL_ORDER= ['COMPRESSING', 'STABLE', 'EXPANDING']

# session windows (minutes since midnight, ET)
IB_START, IB_END, RTH_END = 570, 630, 960       # 09:30, 10:30, 16:00
ORB_ENDS = {'ORB-5': 575, 'ORB-15': 585, 'ORB-30': 600}   # 09:35 / 09:45 / 10:00

print("config loaded | lookback =", REGIME_LOOKBACK, "| baseline =", BASELINE_YEARS, "y")

In [ ]:
# ==================== LOAD + SESSION DAY + REGIME ====================
_df = pd.read_csv(DATA_PATH)
_df['dt'] = pd.to_datetime(_df['Date'] + ' ' + _df['Time'], format='%Y-%m-%d %H:%M:%S')
_df = _df.set_index('dt').sort_index()
_df['sess'] = (_df.index + pd.Timedelta(hours=SESSION_SHIFT_HOURS)).normalize()
_df['tod']  = _df.index.hour * 60 + _df.index.minute
print(f"{len(_df):,} 1-min rows | {_df.index[0]} -> {_df.index[-1]}")

# daily close on the SAME session key -> regime quantities
_daily = _df.groupby('sess')['Close'].last()
_ret   = np.log(_daily / _daily.shift(1))
_trail = max(30, round(BASELINE_YEARS * BARS_PER_YEAR))
_baseSD = _ret.rolling(_trail, min_periods=_trail).std(ddof=0)

def _prank(s, n):
    v = s.to_numpy(float); out = np.full(len(v), np.nan)
    for i in range(n, len(v)):
        if not np.isnan(v[i]): out[i] = np.mean(v[i-n:i] <= v[i]) * 100.0
    return pd.Series(out, index=s.index)

regime = pd.DataFrame(index=_daily.index)
regime['base_sd'] = _baseSD
for L in (10, 20, 50):
    sd = _ret.rolling(L, min_periods=L).std(ddof=0)
    q  = sd / _baseSD.replace(0, np.nan)
    regime[f'sd_{L}'] = sd
    regime[f'ratio_{L}'] = q
    regime[f'pct_{L}']  = _prank(sd, _trail)
    regime[f'trend_{L}'] = q - q.shift(TREND_WINDOW)

L = REGIME_LOOKBACK
tr = regime[f'trend_{L}']
regime['overall'] = np.where(tr.isna(), '--',
                    np.where(tr < -TREND_THRESHOLD, 'COMPRESSING',
                    np.where(tr >  TREND_THRESHOLD, 'EXPANDING', 'STABLE')))
# the active conditioner columns for the chosen lookback
regime['ratio'] = regime[f'ratio_{L}']
regime['pct']   = regime[f'pct_{L}']

_valid = regime['ratio'].notna()
print(f"regime-valid days: {int(_valid.sum())} | "
      f"{regime.index[_valid][0].date()} -> {regime.index[_valid][-1].date()}")

In [ ]:
# ==================== BINNERS + REPORT HELPER ====================
def bin_ratio(s):
    return pd.cut(s, RATIO_EDGES, labels=RATIO_LABELS, right=False)

def bin_pct(s):
    labs = [f'{PCT_EDGES[i]}-{PCT_EDGES[i+1]}' for i in range(len(PCT_EDGES)-1)]
    return pd.cut(s, PCT_EDGES, labels=labs, include_lowest=True)

def bin_overall(s):
    return pd.Categorical(s, categories=OVERALL_ORDER, ordered=True)

CONDITIONERS = {'ratio': bin_ratio, 'percentile': bin_pct, 'overall': bin_overall}

def report(ev, metrics, title, conditioners=('ratio', 'percentile', 'overall'),
           min_n=MIN_N):
    """ev: event table joined to regime (has 'ratio','pct','overall').
    metrics: list of (label, fn, fmt). fn(subdf)->float|None, fmt e.g. '{:.1f}'."""
    valid = ev[ev['ratio'].notna()]
    display(Markdown(f"#### {title}"))
    def line(name, sub, gate=True):
        n = len(sub)
        cells_ = [f"{name:<12}", f"n={n:>5}"]
        if gate and n < min_n:
            cells_.append("(below min sample)")
        else:
            for lab, fn, fmt in metrics:
                v = fn(sub) if n else None
                cells_.append(f"{lab}=" + (fmt.format(v) if v is not None else "  --"))
        print("  ".join(cells_))
    for cond in conditioners:
        col = {'ratio': 'ratio', 'percentile': 'pct', 'overall': 'overall'}[cond]
        b = CONDITIONERS[cond](valid[col])
        print(f"\n  by {cond}:")
        cats = (b.categories if hasattr(b, 'categories') else b.cat.categories)
        bs = pd.Series(np.asarray(b), index=valid.index)
        for c in cats:
            line(str(c), valid[bs == c])
        line('ALL(regime)', valid, gate=False)
        line('ALL(hist)',  ev,    gate=False)
    print()

# metric shorthands
def rate(col):   return lambda d: d[col].mean()*100 if len(d) else None
def med(col):    return lambda d: d[col].median()
def q(col, p):   return lambda d: d[col].quantile(p)
def match_rate(a, b): return lambda d: (d[a]==d[b]).mean()*100 if len(d) else None
PCT='{:.1f}%'; NUM='{:.1f}'; P3='{:.3f}%'
print("helpers ready")

In [ ]:
# ==================== BUILD IB EVENT TABLE (09:30-10:30) ====================
rth = _df[(_df['tod'] >= IB_START) & (_df['tod'] < RTH_END)]
rows = []
for sess, day in rth.groupby('sess'):
    ib = day[day['tod'] < IB_END]; post = day[day['tod'] >= IB_END]
    if len(ib) < 30 or post.empty:
        continue
    o, cl = ib['Open'].iloc[0], ib['Close'].iloc[-1]
    hi, lo = ib['High'].max(), ib['Low'].min()
    rng = hi - lo
    if rng <= 0 or np.isnan([o, cl, hi, lo]).any():
        continue
    mid = (hi + lo) / 2
    hi_t = ib['High'].idxmax(); lo_t = ib['Low'].idxmin()
    H = post['High'].to_numpy(); Lw = post['Low'].to_numpy(); idx = post.index
    hb = H > hi; lb = Lw < lo
    ti_hi = np.argmax(hb) if hb.any() else None
    ti_lo = np.argmax(lb) if lb.any() else None
    def mins(ti): return (idx[ti] - idx[0]).total_seconds()/60 + 1 if ti is not None else np.nan
    # MFE/MAE after each break (%)
    hmfe = hmae = lmfe = lmae = np.nan
    if ti_hi is not None:
        hmfe = (H[ti_hi:].max() - hi)/hi*100
        hmae = (hi - Lw[ti_hi:].min())/hi*100
    if ti_lo is not None:
        lmfe = (lo - Lw[ti_lo:].min())/lo*100
        lmae = (H[ti_lo:].max() - lo)/lo*100
    # midpoint retest (0.1% band)
    thr = mid*0.001; C = post['Close'].to_numpy()
    left = np.where(np.abs(C - mid) > thr)[0]
    mret = False; mret_t = np.nan
    if len(left):
        j0 = left[0]
        near = (np.abs(Lw[j0:] - mid) <= thr) | (np.abs(H[j0:] - mid) <= thr)
        if near.any():
            mret = True
            mret_t = (idx[j0 + np.argmax(near)] - idx[j0]).total_seconds()/60
    rows.append(dict(sess=sess, ib_dir='bull' if cl>o else 'bear',
        close_half='upper' if cl>mid else 'lower',
        formed_first='high' if hi_t<=lo_t else 'low',
        ib_range_pct=rng/o*100,
        high_broke=hb.any(), low_broke=lb.any(),
        both=hb.any() and lb.any(), neither=(not hb.any()) and (not lb.any()),
        first_high=(ti_hi is not None and (ti_lo is None or ti_hi<=ti_lo)),
        t_high=mins(ti_hi), t_low=mins(ti_lo),
        high_mfe=hmfe, high_mae=hmae, low_mfe=lmfe, low_mae=lmae,
        mid_retest=mret, mid_retest_t=mret_t))
IB = pd.DataFrame(rows).set_index('sess').join(regime[['ratio','pct','overall']])
print(f"IB days: {len(IB)} | regime-valid: {int(IB['ratio'].notna().sum())}")

In [ ]:
# ==================== IB ANALYSES ====================
report(IB, [('high',rate('high_broke'),PCT), ('low',rate('low_broke'),PCT),
            ('both',rate('both'),PCT), ('neither',rate('neither'),PCT)],
       "IB break rates (post-IB 10:30-16:00)")

report(IB, [('firstHigh',rate('first_high'),PCT),
            ('t_hi_med',med('t_high'),NUM), ('t_hi_p25',q('t_high',.25),NUM),
            ('t_hi_p75',q('t_high',.75),NUM)],
       "IB time to first break (minutes from 10:30) + which side first")

report(IB, [('bull',rate('ib_dir') if False else (lambda d:(d['ib_dir']=='bull').mean()*100),PCT),
            ('highFirst',rate('formed_first') if False else (lambda d:(d['formed_first']=='high').mean()*100),PCT)],
       "IB direction (bull%) and extreme formed first (high%)")

report(IB, [('Hbrk_MFE',med('high_mfe'),P3), ('Hbrk_MAE',med('high_mae'),P3),
            ('Lbrk_MFE',med('low_mfe'),P3), ('Lbrk_MAE',med('low_mae'),P3)],
       "IB post-break excursion, median % (MFE = follow-through, MAE = adverse)")

report(IB, [('retest%',rate('mid_retest'),PCT), ('t_med',med('mid_retest_t'),NUM)],
       "IB midpoint retest rate + median minutes to retest")

In [ ]:
# ==================== IB 2-WAY: regime(ratio) x IB close-half ====================
display(Markdown("#### IB break rates by ratio bin x IB close-half"))
v = IB[IB['ratio'].notna()].copy()
v['rbin'] = bin_ratio(v['ratio'])
print(f"  {'ratio bin':<10}{'close':<7}{'n':>6}{'high%':>8}{'low%':>8}{'both%':>8}{'neither%':>10}")
for rb in RATIO_LABELS:
    for ch in ['upper','lower']:
        s = v[(v['rbin']==rb) & (v['close_half']==ch)]
        n=len(s)
        if n==0: continue
        tag='' if n>=MIN_N else '  (<MIN_N)'
        print(f"  {rb:<10}{ch:<7}{n:>6}{s['high_broke'].mean()*100:>8.1f}"
              f"{s['low_broke'].mean()*100:>8.1f}{s['both'].mean()*100:>8.1f}"
              f"{s['neither'].mean()*100:>10.1f}{tag}")

In [ ]:
# ==================== BUILD ORB EVENT TABLES (5/15/30) ====================
def build_orb(end_tod):
    seg = _df[(_df['tod'] >= IB_START) & (_df['tod'] < RTH_END)]
    rows = []
    for sess, day in seg.groupby('sess'):
        orb = day[day['tod'] < end_tod]; rest = day[day['tod'] >= end_tod]
        if len(orb) < 3 or rest.empty:
            continue
        o, cl = orb['Open'].iloc[0], orb['Close'].iloc[-1]
        hi, lo = orb['High'].max(), orb['Low'].min()
        if hi <= lo: continue
        mid = (hi+lo)/2; is_bull = cl > o
        ny_open = day['Open'].iloc[0]; ny_close = day['Close'].iloc[-1]
        ny_dir = 'bull' if ny_close > ny_open else 'bear'
        H = rest['High'].to_numpy(); Lw = rest['Low'].to_numpy(); idx = rest.index
        hb = H > hi; lb = Lw < lo
        ti_hi = np.argmax(hb) if hb.any() else None
        ti_lo = np.argmax(lb) if lb.any() else None
        if ti_hi is None and ti_lo is None: first='none'
        elif ti_lo is None or (ti_hi is not None and ti_hi<=ti_lo): first='high'
        else: first='low'
        fb_t = np.nan
        if first=='high': fb_t=(idx[ti_hi]-idx[0]).total_seconds()/60+1
        elif first=='low': fb_t=(idx[ti_lo]-idx[0]).total_seconds()/60+1
        # retest of breakout level + continuation (next 5 bars), ORB-5 style
        retest=False; cont=np.nan; rmfe=rmae=np.nan
        if first in ('high','low'):
            lvl = hi if first=='high' else lo
            bti = ti_hi if first=='high' else ti_lo
            aH=H[bti+1:]; aL=Lw[bti+1:]; tol=lvl*0.0002   # scan AFTER the break bar
            for k in range(len(aH)):
                touch = (abs(aL[k]-lvl)<tol) if first=='high' else (abs(aH[k]-lvl)<tol)
                inside = (aL[k]<=lvl<=aH[k])
                if touch or inside:
                    retest=True
                    if first=='high':
                        rmfe=(aH[k:].max()-lvl)/lvl*100; rmae=(lvl-aL[k:].min())/lvl*100
                    else:
                        rmfe=(lvl-aL[k:].min())/lvl*100; rmae=(aH[k:].max()-lvl)/lvl*100
                    nxt=slice(k+1,k+6)
                    if first=='high':
                        if aH[nxt].size and aH[nxt].max()>lvl*1.0005: cont=True
                        elif aL[nxt].size and aL[nxt].min()<lvl*0.9995: cont=False
                    else:
                        if aL[nxt].size and aL[nxt].min()<lvl*0.9995: cont=True
                        elif aH[nxt].size and aH[nxt].max()>lvl*1.0005: cont=False
                    break
        rows.append(dict(sess=sess, is_bull=is_bull, orb_dir='bull' if is_bull else 'bear',
            ny_dir=ny_dir, range_pct=(hi-lo)/lo*100,
            high_broke=hb.any(), low_broke=lb.any(),
            both=hb.any() and lb.any(), neither=(not hb.any()) and (not lb.any()),
            first=first, first_high=(first=='high'), fb_t=fb_t,
            retest=retest, cont=cont, r_mfe=rmfe, r_mae=rmae))
    d = pd.DataFrame(rows).set_index('sess').join(regime[['ratio','pct','overall']])
    d['win'] = d['orb_dir'] == d['ny_dir']
    return d

ORB = {name: build_orb(end) for name, end in ORB_ENDS.items()}
for k,v in ORB.items(): print(f"{k}: {len(v)} days | regime-valid {int(v['ratio'].notna().sum())}")

In [ ]:
# ==================== ORB ANALYSES (5 / 15 / 30) ====================
for name, d in ORB.items():
    display(Markdown(f"### {name}  (09:30-{ {'ORB-5':'09:35','ORB-15':'09:45','ORB-30':'10:00'}[name] })"))
    report(d, [('high',rate('high_broke'),PCT), ('low',rate('low_broke'),PCT),
               ('firstHigh',rate('first_high'),PCT), ('neither',rate('neither'),PCT),
               ('fb_med',med('fb_t'),NUM)],
           f"{name} break rates + first-break side + median break minute")
    report(d, [('win%',rate('win'),PCT)],
           f"{name} direction -> NY session match (win rate)")

In [ ]:
# ==================== ORB-5 retest + excursion ====================
d = ORB['ORB-5']
report(d, [('retest%',rate('retest'),PCT),
           ('cont%',(lambda s: s['cont'].mean()*100 if s['cont'].notna().any() else None),PCT),
           ('r_MFE',med('r_mfe'),P3), ('r_MAE',med('r_mae'),P3)],
       "ORB-5 breakout-level retest rate, continuation rate, and excursion from retest")

In [ ]:
# ==================== COLLINEARITY CHECK ====================
# If regime bucket and same-day ORB/IB range are nearly the same axis, the
# "regime effect" is partly just range size. Report the correlation plainly.
v = IB[IB['ratio'].notna()]
o5 = ORB['ORB-5']; o5 = o5[o5['ratio'].notna()]
print("Spearman corr of same-day range% with the regime conditioners:")
for label, ev, rng in [('IB range%', v, 'ib_range_pct'), ('ORB-5 range%', o5, 'range_pct')]:
    cr = ev[[rng,'ratio']].corr(method='spearman').iloc[0,1]
    cp = ev[[rng,'pct']].corr(method='spearman').iloc[0,1]
    print(f"  {label:<12}  vs ratio: {cr:+.2f}   vs percentile: {cp:+.2f}")
print("\n(High positive corr means 'bigger range' and 'higher regime bin' overlap;"
      "\n read break/excursion moves partly as size, not pure regime.)")

---
# Structural ORB / IB stats — conditioned on regime

Ports the structural engine (ORB = 09:30–09:44, IB = 09:30–10:29, full-RTH days
only) and reads each measure **per regime bin**. The engine reuses the `_df`,
`regime`, `report()` and binners already defined above — no second load.

**Why the join is valid:** the source script keys its day by calendar midnight;
we key by the `+6h` session. For every RTH bar (09:30–16:59) those two are the
same date (they only diverge at the 18:00 overnight open), so its `days` table
joins to `regime` correctly.

Four measures get the regime treatment — the ones with a real volatility prior:
- **extreme survival** — does the 15-min ORB high/low remain the IB high/low
- **retracement depth** — how far price pulls back before the IB extreme forms
- **invalidation** — ORB mid / held-extreme violation after the break
- **post-IB** — IB-break continuation and extension (in IB-range units)

`ib_mfe_x` (extension in units of that day's IB range) is already self-scaled,
so it's the regime metric least contaminated by "vol is vol".

In [ ]:
# ==================== STRUCTURAL ENGINE (lean, reuses _df) ====================
# ORB 09:30-09:44, IB 09:30-10:29, RTH 09:30-15:59 ; full 390-bar days only.
_ORB_END, _IB_END, _RTH_CLOSE, _FULL = 585, 630, 960, 390   # minutes-of-day

_S = _df[['Open','High','Low','Close']].rename(columns=str.lower).copy()
_S['mod'] = _S.index.hour*60 + _S.index.minute
_S['day'] = _S.index.normalize()           # == +6h session date for RTH bars

def _first_break(h, l, mods, hi, lo, from_mod):
    m = mods >= from_mod
    up = np.flatnonzero(m & (h > hi)); dn = np.flatnonzero(m & (l < lo))
    fu = up[0] if len(up) else np.inf;  fd = dn[0] if len(dn) else np.inf
    if fu == np.inf and fd == np.inf: return "none", np.nan
    if fu == fd: return "both", mods[int(fu)]
    return ("up", mods[int(fu)]) if fu < fd else ("down", mods[int(fd)])

def build_days_struct(S):
    rth = S[(S['mod'] >= IB_START) & (S['mod'] < _RTH_CLOSE)]
    cnt = rth.groupby('day').size(); good = set(cnt[cnt == _FULL].index)
    rows = []
    for day, g in rth.groupby('day'):
        if day not in good: continue
        h=g['high'].to_numpy(); l=g['low'].to_numpy(); c=g['close'].to_numpy()
        o=g['open'].to_numpy(); mods=g['mod'].to_numpy()
        io=np.flatnonzero(mods<_ORB_END); ii=np.flatnonzero(mods<_IB_END)
        ohi,olo=h[io].max(),l[io].min(); ihi,ilo=h[ii].max(),l[ii].min()
        omid=(ohi+olo)/2; imid=(ihi+ilo)/2
        o_mhi=mods[io][np.argmax(h[io])]; o_mlo=mods[io][np.argmin(l[io])]
        i_mhi=mods[ii][np.argmax(h[ii])]; i_mlo=mods[ii][np.argmin(l[ii])]
        oc=c[io][-1]; ic=c[ii][-1]
        obrk,obrk_mod=_first_break(h,l,mods,ohi,olo,_ORB_END)
        ibrk,ibrk_mod=_first_break(h,l,mods,ihi,ilo,_IB_END)
        rows.append(dict(day=day, year=day.year,
            ohi=ohi, olo=olo, orng=ohi-olo, omid=omid,
            o_first='high' if o_mhi<o_mlo else ('low' if o_mlo<o_mhi else 'same'),
            o_half='upper' if oc>omid else ('lower' if oc<omid else 'mid'),
            obrk=obrk, obrk_mod=obrk_mod,
            ihi=ihi, ilo=ilo, irng=ihi-ilo, imid=imid,
            i_first='high' if i_mhi<i_mlo else ('low' if i_mlo<i_mhi else 'same'),
            ibrk=ibrk, ibrk_mod=ibrk_mod, i_mhi=i_mhi, i_mlo=i_mlo,
            orb_hi_is_ib_hi=bool(ohi==ihi), orb_lo_is_ib_lo=bool(olo==ilo)))
    D = pd.DataFrame(rows).set_index('day')
    D['surv_both']    = D['orb_hi_is_ib_hi'] & D['orb_lo_is_ib_lo']
    D['surv_neither'] = ~D['orb_hi_is_ib_hi'] & ~D['orb_lo_is_ib_lo']
    return D

def build_paths_struct(S, D):
    rth = S[(S['mod'] >= IB_START) & (S['mod'] < _RTH_CLOSE)]
    recs = []
    for day, g in rth.groupby('day'):
        if day not in D.index: continue
        r = D.loc[day]
        h=g['high'].to_numpy(); l=g['low'].to_numpy(); c=g['close'].to_numpy()
        mods=g['mod'].to_numpy(); rec=dict(day=day)
        # ORB break leg
        if r['obrk'] in ('up','down') and r['obrk_mod'] < _IB_END:
            up = r['obrk']=='up'; brk = r['ohi'] if up else r['olo']
            held = r['olo'] if up else r['ohi']; after = mods > r['obrk_mod']
            def fm(mask):
                idx=np.flatnonzero(after & mask); return mods[idx[0]] if len(idx) else np.nan
            rec['mid_loss_mod']  = fm(l<=r['omid'] if up else h>=r['omid'])
            rec['held_loss_mod'] = fm(l<=held if up else h>=held)
            t_ext = r['i_mhi'] if up else r['i_mlo']
            if t_ext > r['obrk_mod']:
                w = after & (mods<=t_ext)
                if w.any():
                  deep = l[w].min() if up else h[w].max()
                  if (deep>r['ohi']) if up else (deep<r['olo']):   zone='no retrace'
                  elif (deep>=r['omid']) if up else (deep<=r['omid']): zone='near half'
                  elif (deep>=r['olo']) if up else (deep<=r['ohi']):   zone='far half'
                  else: zone='through held side'
                  rec['retrace_zone']=zone
                  rec['move_to_ib_extreme']=abs((r['ihi'] if up else r['ilo'])-brk)/r['orng']
        # IB break leg
        if r['ibrk'] in ('up','down'):
            up = r['ibrk']=='up'; brk = r['ihi'] if up else r['ilo']
            held = r['ilo'] if up else r['ihi']; after = mods > r['ibrk_mod']
            rec['ib_cont'] = (r['obrk']==r['ibrk'])
            rec['ib_close_beyond'] = bool(c[-1]>brk) if up else bool(c[-1]<brk)
            if after.any():                       # empty when IB breaks on the last bar
                rec['ib_held_holds'] = not bool((l[after]<=held).any() if up else (h[after]>=held).any())
                mfe = (h[after].max()-brk) if up else (brk-l[after].min())
                rec['ib_mfe_x'] = mfe/r['irng'] if r['irng']>0 else np.nan
            else:
                rec['ib_held_holds'] = True; rec['ib_mfe_x'] = 0.0
        recs.append(rec)
    return pd.DataFrame(recs).set_index('day')

DAYS  = build_days_struct(_S)
PATHS = build_paths_struct(_S, DAYS)
REG3  = regime[['ratio','pct','overall']]
print(f"structural days: {len(DAYS)} full-RTH | regime-valid: {int(DAYS.join(REG3)['ratio'].notna().sum())}")

In [ ]:
# ==================== REGIME: ORB extreme survival into the IB ====================
ev = DAYS.join(REG3)
report(ev, [('ORBhi=IBhi',rate('orb_hi_is_ib_hi'),PCT),
            ('ORBlo=IBlo',rate('orb_lo_is_ib_lo'),PCT),
            ('both',rate('surv_both'),PCT), ('neither',rate('surv_neither'),PCT)],
       "Does the 15-min ORB extreme survive as the IB extreme? (prior: survival falls as vol rises)")

In [ ]:
# ==================== REGIME: retracement depth before IB extreme ====================
ev = DAYS.join(PATHS[['retrace_zone']]).join(REG3)
ev = ev[ev['retrace_zone'].notna()].copy()
for z in ['no retrace','near half','far half','through held side']:
    ev[f'z_{z}'] = (ev['retrace_zone']==z)
report(ev, [('none',rate('z_no retrace'),PCT), ('near',rate('z_near half'),PCT),
            ('far',rate('z_far half'),PCT), ('thruHeld',rate('z_through held side'),PCT)],
       "Deepest pullback before the IB extreme forms (ORB-break days)")

In [ ]:
# ==================== REGIME: invalidation after the ORB break ====================
ev = DAYS.join(PATHS[['mid_loss_mod','held_loss_mod']]).join(REG3)
ev = ev[ev['obrk'].isin(['up','down']) & (ev['obrk_mod'] < _IB_END)].copy()
ev['mid_lost']  = ev['mid_loss_mod'].notna()
ev['held_lost'] = ev['held_loss_mod'].notna()
ev['mid_lost_min'] = ev['mid_loss_mod'] - ev['obrk_mod']
report(ev, [('midLost%',rate('mid_lost'),PCT), ('heldLost%',rate('held_lost'),PCT),
            ('t_mid_med',med('mid_lost_min'),NUM)],
       "Post-break invalidation: trades back through ORB mid / through the held extreme")

In [ ]:
# ==================== REGIME: post-IB break continuation & extension ====================
ev = DAYS.join(PATHS[['ib_cont','ib_held_holds','ib_mfe_x','ib_close_beyond']]).join(REG3)
ev = ev[ev['ibrk'].isin(['up','down'])].copy()
ev['reach1x'] = ev['ib_mfe_x'] >= 1.0
report(ev, [('cont%',rate('ib_cont'),PCT), ('heldHolds%',rate('ib_held_holds'),PCT),
            ('reach1xIB%',rate('reach1x'),PCT), ('ext_med(xIB)',med('ib_mfe_x'),'{:.2f}')],
       "Post-IB break: continuation vs ORB, held-side survival, extension in IB-range units")

---
# Baseline sweep — 1y / 2y / 5y / 10y

The event tables (IB, ORB, structural) don't depend on the baseline, so only the
regime table is recomputed for each. Shorter baselines finish warm-up earlier and
give many more regime-valid days:

| baseline | trail bars | ~valid days |
|---|---|---|
| 1y | 252 | ~3,500 (from ~2011) |
| 2y | 504 | ~3,300 (from ~2012) |
| 5y | 1,260 | ~2,500 (from ~2015) |
| 10y | 2,520 | ~1,280 (from 2020-03) |

**Conditioner = percentile deciles** (stable across baselines). Ratio changes
meaning as the baseline shortens — `baseSD` chases the recent past, so ratio
mean-reverts toward 1 — so it isn't comparable across baselines the way percentile is.

For each measure: **spread** = (max − min) rate across deciles with n ≥ MIN_N.
**null95** = 95th-percentile spread from 200 permutations of the regime labels.
A `*` means the real spread beats the null (a regime effect); no star = flat / noise.

In [ ]:
# ==================== recompute regime for each baseline ====================
def regime_for(years):
    trail = max(30, round(years * BARS_PER_YEAR))
    baseSD = _ret.rolling(trail, min_periods=trail).std(ddof=0)
    L = REGIME_LOOKBACK
    sd = _ret.rolling(L, min_periods=L).std(ddof=0)
    q  = sd / baseSD.replace(0, np.nan)
    p  = _prank(sd, trail)
    tr = q - q.shift(TREND_WINDOW)
    ov = np.where(tr.isna(), '--',
         np.where(tr < -TREND_THRESHOLD, 'COMPRESSING',
         np.where(tr >  TREND_THRESHOLD, 'EXPANDING', 'STABLE')))
    return pd.DataFrame({'ratio': q, 'pct': p, 'overall': ov}, index=_ret.index)

BASELINES = [1, 2, 5, 10]
REGIMES = {y: regime_for(y) for y in BASELINES}
print("regime-valid days by baseline:")
for y in BASELINES:
    r = REGIMES[y]; v = r['pct'].notna()
    print(f"  {y:>2}y : {int(v.sum()):>5} days   from {r.index[v][0].date()}")

In [ ]:
# ==================== spread-vs-null across baselines ====================
def _spread_and_null(y_bool, pct, n_perm=200, min_n=MIN_N, seed=7):
    """max-min decile rate (bins with n>=min_n) + 95th pct of permuted spreads."""
    yb = np.asarray(y_bool, dtype=float); pct = np.asarray(pct, dtype=float)
    m = np.isfinite(pct) & np.isfinite(yb)
    y = yb[m]; pc = pct[m]
    b = np.clip((pc // 10).astype(int), 0, 9)               # decile 0..9
    cnt = np.bincount(b, minlength=10)
    ok = cnt >= min_n
    if ok.sum() < 2: return np.nan, np.nan, int(m.sum()), int(ok.sum())
    def spread(yy):
        s = np.bincount(b, weights=yy, minlength=10)
        rate = np.where(cnt > 0, s / np.maximum(cnt, 1), np.nan)
        r = rate[ok]; return np.nanmax(r) - np.nanmin(r)
    obs = spread(y)
    rng = np.random.default_rng(seed)
    perm = np.array([spread(rng.permutation(y)) for _ in range(n_perm)])
    return obs * 100, np.nanpercentile(perm, 95) * 100, int(m.sum()), int(ok.sum())

# build the event frames whose boolean columns we test
_orb15 = ORB['ORB-15']
_inval = DAYS.join(PATHS[['mid_loss_mod']]); _inval = _inval[_inval['obrk'].isin(['up','down']) & (_inval['obrk_mod']<_IB_END)]
_inval = _inval.assign(mid_lost=_inval['mid_loss_mod'].notna())
_cont  = DAYS.join(PATHS[['ib_cont']]); _cont = _cont[_cont['ibrk'].isin(['up','down'])]
_retr  = DAYS.join(PATHS[['retrace_zone']]); _retr = _retr[_retr['retrace_zone'].notna()]
_retr  = _retr.assign(near=_retr['retrace_zone']=='near half')

# (label, event_df, boolean-series)  — all probability/odds measures
MEAS = [
  ('IB break high',      IB,      IB['high_broke']),
  ('IB break low',       IB,      IB['low_broke']),
  ('IB neither',         IB,      IB['neither']),
  ('IB high-first',      IB,      IB['first_high']),
  ('IB dir=bull',        IB,      IB['ib_dir']=='bull'),
  ('IB mid-retest',      IB,      IB['mid_retest']),
  ('ORB15 break high',   _orb15,  _orb15['high_broke']),
  ('ORB15->NY win',      _orb15,  _orb15['win']),
  ('ORBhi=IBhi (surv)',  DAYS,    DAYS['orb_hi_is_ib_hi']),
  ('surv neither',       DAYS,    DAYS['surv_neither']),
  ('retrace near-half',  _retr,   _retr['near']),
  ('ORB mid lost',       _inval,  _inval['mid_lost']),
  ('post-IB cont',       _cont,   _cont['ib_cont']),
]

hdr = f"{'measure':<20}" + "".join(f"{str(y)+'y':>14}" for y in BASELINES)
print(hdr); print("-"*len(hdr))
for label, df, ycol in MEAS:
    cells=[f"{label:<20}"]
    for y in BASELINES:
        pct = REGIMES[y]['pct'].reindex(df.index).to_numpy()
        obs, nul, nv, nbins = _spread_and_null(ycol, pct)
        if np.isnan(obs): cells.append(f"{'--':>14}"); continue
        star = '*' if obs > nul else ' '
        cells.append(f"{obs:>6.1f}/{nul:>4.1f}{star:>2}")
    print("".join(cells))
print("\n  cell = spread / null95 (pp). '*' = real spread beats the shuffle null.")
print("  spread = max-min decile rate; deciles with n<MIN_N dropped.")

In [ ]:
# ==================== contrast: a MAGNITUDE measure across baselines ====================
# Odds are flat; magnitude should scale. IB high-break MFE (median %) per decile.
def _mfe_spread(vals, pct, min_n=MIN_N):
    m = np.isfinite(pct) & np.isfinite(vals)
    v = vals[m]; b = np.clip((pct[m]//10).astype(int),0,9)
    meds=[]; 
    for d in range(10):
        vv=v[b==d]
        if len(vv)>=min_n: meds.append(np.median(vv))
    return (max(meds)-min(meds), min(meds), max(meds)) if len(meds)>=2 else (np.nan,np.nan,np.nan)

print(f"{'':<22}" + "".join(f"{str(y)+'y':>18}" for y in BASELINES))
print(f"{'IB Hbrk MFE median %':<22}" + "".join(
    (lambda s: f"{s[1]:>7.3f}-{s[2]:<7.3f}" if np.isfinite(s[0]) else f"{'--':>18}")(
        _mfe_spread(IB['high_mfe'].to_numpy(),
                    REGIMES[y]['pct'].reindex(IB.index).to_numpy()))
    for y in BASELINES))
print("\n  low-decile median  ->  high-decile median. Widens with vol at every baseline:")
print("  magnitude scales (mechanical); the odds table above stays flat.")